In [1]:
#!/usr/bin/env python3
# =============================================================================
# ML-Based IDS for IPv6 ND & RA Attack Detection — v17 STRATIFIED
#
# ── CHANGES FROM v17 ORIGINAL ────────────────────────────────────────────────
#
#  REMOVED: Chronological split (80/20 time-ordered)
#  REMOVED: Temporal holdout split (65/35 time-ordered)
#  REPLACED WITH:
#    - Stratified 80/20 holdout (Split A) — random_state=42
#    - Stratified 80/20 holdout (Split B) — random_state=123 (different partition)
#    - 5-Fold Stratified CV for stability check (replaces temporal robustness)
#
#  All other v17 fixes retained:
#  FIX-AE  SMOTE oversampling on training data only
#  FIX-AF  Enhanced XGBoost regularization (gamma=1.0, reg_lambda=2.0)
#  FIX-AG  safe_predict_proba() for all models
#  FIX-Z   Top-10 features (ANOVA-F, train-only)
#  FIX-AA  Hyperparameter sweeps
#  FIX-AB  Feature-count sensitivity
#  FIX-AC  _SafeXGB label-safe wrapper
#  FIX-AD  StratifiedKFold for HP sweep
#  FIX-VIZ Legend fix, 11 output files
# =============================================================================

import warnings; warnings.filterwarnings("ignore")
import os;       os.environ["LOKY_MAX_CPU_COUNT"] = "1"
import math, csv, sys, json
from pathlib import Path
from collections import defaultdict, Counter

import numpy  as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot  as plt
import matplotlib.ticker  as mticker
import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap

from scapy.all import PcapReader, IPv6, ICMPv6ND_RA, ICMPv6ND_NS, ICMPv6ND_NA, Ether
from scapy.all import ICMPv6EchoRequest, ICMPv6EchoReply

from sklearn.dummy            import DummyClassifier
from sklearn.base             import BaseEstimator, ClassifierMixin
from sklearn.decomposition    import PCA
from sklearn.ensemble         import RandomForestClassifier
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.inspection       import permutation_importance
from sklearn.model_selection  import (StratifiedKFold, train_test_split,
                                       learning_curve, cross_validate)
from sklearn.preprocessing    import LabelEncoder, StandardScaler, label_binarize
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score,
    precision_score, recall_score,
    roc_auc_score, average_precision_score,
    classification_report, confusion_matrix,
    ConfusionMatrixDisplay,
)

try:
    from xgboost import XGBClassifier;  _HAS_XGB = True
except ImportError:
    _HAS_XGB = False
    print("[WARN] xgboost not installed — XGBoost is co-primary in v17.")

try:
    from imblearn.over_sampling import SMOTE;  _HAS_SMOTE = True
except ImportError:
    _HAS_SMOTE = False
    print("[WARN] imblearn not installed. Install with: pip install imbalanced-learn")
    print("       SMOTE oversampling will be skipped.")

try:
    from lightgbm import LGBMClassifier;     _HAS_LGB = True
except ImportError:
    _HAS_LGB = False

try:
    from catboost import CatBoostClassifier; _HAS_CAT = True
except ImportError:
    _HAS_CAT = False

# ─────────────────────────────────────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────────────────────────────────────
PCAP_PATH       = "./raw_capture.pcap"
NODES_CSV_PATH  = "./raw_capture_nodes.csv"
EVENTS_CSV_PATH = "./raw_capture_events.csv"
CSV_PKT_PATH    = "./ipv6_ids_packets_v17.csv"
CSV_WIN_PATH    = "./ipv6_ids_windows_v17.csv"

RANDOM_STATE    = 42
RANDOM_STATE_B  = 123          # Different seed for Split B (second stratified partition)
N_FOLDS         = 5
TEST_SIZE       = 0.20

LOCKED_TOP5_REF = [
    "ns_burst_rate", "unique_src_count", "sliding_nd_ratio_3s",
    "ns_unique_tgt_r", "multicast_ratio",
]
PRIMARY_K = 10

XGB_LR_SWEEP    = [0.05, 0.1, 0.2, 0.3]
XGB_DEPTH_SWEEP = [3, 4, 5, 6]

RF_NEST_SWEEP   = [200, 300, 500]
RF_DEPTH_SWEEP  = [5, 7, 10, None]
K_FEAT_SWEEP    = [5, 10, 15]

RA_GLOBAL_THRESHOLD  = 5
NS_GLOBAL_THRESHOLD  = 100
MIN_CLASS_WINDOWS    = 10
LEAKAGE_AUC_WARN     = 0.98
OVERFIT_GAP_WARN     = 0.05
MAX_SINGLE_FEAT_BACC = 0.999
BOOTSTRAP_N_ITER     = 500
ALWAYS_EXCLUDED_FEATURES = {"ra_rate", "ns_rate"}
MIN_FOLD_TRAIN  = 30
BURST_SUBWINDOW = 0.25

# ─────────────────────────────────────────────────────────────────────────────
# PLOT THEME
# ─────────────────────────────────────────────────────────────────────────────
DARK_BG  = "#0d1117";  PANEL_BG = "#161b22";  CARD_BG = "#1c2333"
ACCENT1  = "#e94560";  ACCENT2  = "#4cc9f0";  ACCENT3 = "#7b2fbe"
ACCENT4  = "#f4a261";  ACCENT5  = "#2ec4b6"
TEXT_COL = "#e6edf3";  GRID_COL = "#21262d";  BORDER  = "#30363d"

CLS_COLORS = {"Normal": "#4cc9f0", "RA_Attack": "#e94560",
              "ND_Attack": "#7b2fbe", "Combined_Attack": "#f4a261"}

LEG_KW = dict(labelcolor=TEXT_COL, facecolor=CARD_BG, edgecolor=BORDER,
              framealpha=1.0, fontsize=9)

def style_ax(ax):
    ax.set_facecolor(PANEL_BG)
    ax.tick_params(colors=TEXT_COL, which="both")
    ax.xaxis.label.set_color(TEXT_COL); ax.yaxis.label.set_color(TEXT_COL)
    ax.title.set_color(TEXT_COL)
    for sp in ax.spines.values(): sp.set_edgecolor(BORDER)
    ax.grid(color=GRID_COL, alpha=0.6, linewidth=0.6)

def savefig(fig, path):
    fig.savefig(path, dpi=150, bbox_inches="tight", facecolor=DARK_BG)
    plt.close(fig); print(f"  Saved → {path}")


def phase_name_to_label(phase_name: str) -> str:
    p = phase_name.lower()
    if "combined" in p: return "Combined_Attack"
    if ("ra" in p or "router" in p) and ("attack" in p or "slow" in p or "flood" in p):
        return "RA_Attack"
    if ("nd" in p or "ns" in p or "solicitat" in p) and ("attack" in p or "slow" in p):
        return "ND_Attack"
    return "Normal"


# ─────────────────────────────────────────────────────────────────────────────
# FACTORY & HELPERS
# ─────────────────────────────────────────────────────────────────────────────

class _SafeXGB(BaseEstimator, ClassifierMixin):
    """Sklearn-compatible XGBClassifier with label re-encoding
    and enhanced regularization (gamma)."""
    def __init__(self, n_estimators=200, max_depth=4, learning_rate=0.1,
                 eval_metric="mlogloss", reg_alpha=0.1, reg_lambda=2.0,
                 gamma=1.0, colsample_bytree=0.8, subsample=0.8,
                 random_state=42, verbosity=0):
        self.n_estimators     = n_estimators
        self.max_depth        = max_depth
        self.learning_rate    = learning_rate
        self.eval_metric      = eval_metric
        self.reg_alpha        = reg_alpha
        self.reg_lambda       = reg_lambda
        self.gamma            = gamma
        self.colsample_bytree = colsample_bytree
        self.subsample        = subsample
        self.random_state     = random_state
        self.verbosity        = verbosity

    def fit(self, X, y):
        self._le = LabelEncoder()
        y_enc = self._le.fit_transform(y)
        self._xgb = XGBClassifier(
            n_estimators=self.n_estimators, max_depth=self.max_depth,
            learning_rate=self.learning_rate, eval_metric=self.eval_metric,
            reg_alpha=self.reg_alpha, reg_lambda=self.reg_lambda,
            gamma=self.gamma, colsample_bytree=self.colsample_bytree,
            subsample=self.subsample, random_state=self.random_state,
            verbosity=self.verbosity,
        )
        self._xgb.fit(X, y_enc)
        self.classes_ = self._le.classes_
        return self

    def predict(self, X):
        return self._le.inverse_transform(self._xgb.predict(X))

    def predict_proba(self, X):
        return self._xgb.predict_proba(X)

    def _get_global_classes(self):
        return self._le.classes_

    @property
    def feature_importances_(self):
        return self._xgb.feature_importances_


def make_xgb(learning_rate=0.1, max_depth=4):
    return _SafeXGB(
        n_estimators=200, max_depth=max_depth, learning_rate=learning_rate,
        eval_metric="mlogloss", reg_alpha=0.1, reg_lambda=2.0,
        gamma=1.0, colsample_bytree=0.8, subsample=0.8,
        random_state=RANDOM_STATE, verbosity=0,
    )


def make_rf(n_estimators=200, max_depth=5):
    return RandomForestClassifier(
        n_estimators=n_estimators, max_depth=max_depth, max_features="sqrt",
        min_samples_leaf=3, class_weight="balanced_subsample",
        random_state=RANDOM_STATE, n_jobs=1,
    )


def safe_predict_proba(clf, X, global_n_cls):
    """Expand any model's predict_proba to global_n_cls columns."""
    raw = clf.predict_proba(X)
    if hasattr(clf, '_get_global_classes'):
        learned_classes = clf._get_global_classes()
    elif hasattr(clf, 'classes_'):
        learned_classes = clf.classes_
    else:
        return raw if raw.shape[1] >= global_n_cls else raw
    n_learned = len(learned_classes)
    if raw.shape[1] == global_n_cls and n_learned == global_n_cls:
        return raw
    full = np.zeros((X.shape[0], global_n_cls))
    for local_i, global_cls in enumerate(learned_classes):
        if local_i < raw.shape[1] and global_cls < global_n_cls:
            full[:, global_cls] = raw[:, local_i]
    return full


def max_burst_rate(times, window=1.0):
    if len(times) < 2: return 0.0
    times = sorted(times); max_r = lo = 0
    for hi in range(len(times)):
        while times[hi] - times[lo] > window: lo += 1
        max_r = max(max_r, (hi - lo + 1) / window)
    return max_r


def safe_roc_auc(y_true, y_prob, nc):
    try:
        present = np.unique(y_true)
        if len(present) < 2: return float("nan"), "skipped (only 1 class)"
        if nc == 2: return float(roc_auc_score(y_true, y_prob[:, 1])), "binary"
        if len(present) < nc:
            scores = []
            for ci in present:
                yb = (np.asarray(y_true) == ci).astype(int)
                if yb.sum() == 0 or (1-yb).sum() == 0: continue
                if ci < y_prob.shape[1]:
                    scores.append(roc_auc_score(yb, y_prob[:, ci]))
            val = float(np.mean(scores)) if scores else float("nan")
            return val, f"partial OvR ({len(present)}/{nc} classes)"
        return float(roc_auc_score(y_true, y_prob, multi_class="ovr",
                                   average="macro")), "macro OvR"
    except Exception as e:
        return float("nan"), f"error: {e}"


def safe_pr_auc(y_true, y_prob, nc):
    try:
        present = np.unique(y_true)
        if len(present) < 2: return float("nan")
        y_bin = label_binarize(y_true, classes=list(range(nc)))
        if y_bin.shape[1] != nc:
            y_bin = np.zeros((len(y_true), nc), dtype=int)
            for i in range(nc): y_bin[:, i] = (np.asarray(y_true) == i).astype(int)
        scores = [average_precision_score(y_bin[:, i], y_prob[:, i])
                  for i in range(nc)
                  if np.sum(y_bin[:, i]) > 0 and np.sum(1-y_bin[:, i]) > 0
                  and i < y_prob.shape[1]]
        return float(np.mean(scores)) if scores else float("nan")
    except Exception as e:
        print(f"  [WARN] PR-AUC: {e}"); return float("nan")


def bootstrap_f1_ci(y_true, y_pred, class_names, n_boot=BOOTSTRAP_N_ITER, ci=0.95):
    rng_b = np.random.default_rng(RANDOM_STATE)
    n = len(y_true); y_true = np.asarray(y_true); y_pred = np.asarray(y_pred)
    n_c = len(class_names); macro_buf = []; class_bufs = {c: [] for c in class_names}
    for _ in range(n_boot):
        idx = rng_b.integers(0, n, size=n)
        yt, yp = y_true[idx], y_pred[idx]
        macro_buf.append(f1_score(yt, yp, average="macro", zero_division=0))
        per = f1_score(yt, yp, average=None, labels=list(range(n_c)), zero_division=0)
        for ci_i, cls in enumerate(class_names):
            class_bufs[cls].append(per[ci_i] if ci_i < len(per) else 0.0)
    lo_p, hi_p = (1-ci)/2*100, (1+ci)/2*100
    result = {}
    for cls in class_names:
        a = np.array(class_bufs[cls])
        result[cls] = (float(np.mean(a)), float(np.percentile(a, lo_p)),
                       float(np.percentile(a, hi_p)))
    a = np.array(macro_buf)
    result["macro"] = (float(np.mean(a)), float(np.percentile(a, lo_p)),
                       float(np.percentile(a, hi_p)))
    return result


def safe_classification_report(y_true, y_pred, le):
    present = sorted(set(np.unique(y_true)) | set(np.unique(y_pred)))
    names = [le.classes_[i] for i in present]
    return classification_report(y_true, y_pred, labels=present,
                                 target_names=names, zero_division=0)


def print_per_class_f1(y_true, y_pred, le, tag=""):
    classes = list(le.classes_); n_c = len(classes); labels = list(range(n_c))
    precs = precision_score(y_true, y_pred, labels=labels, average=None, zero_division=0)
    recs  = recall_score(   y_true, y_pred, labels=labels, average=None, zero_division=0)
    f1s   = f1_score(       y_true, y_pred, labels=labels, average=None, zero_division=0)
    print(f"  ── Per-class metrics {tag} {'─'*max(0,40-len(tag))}")
    print(f"  {'Class':12s}  {'Precision':>10}  {'Recall':>8}  {'F1':>8}")
    print(f"  {'-'*12}  {'-'*10}  {'-'*8}  {'-'*8}")
    for i, cls in enumerate(classes):
        flag = "  ← LOW" if recs[i] < 0.60 and cls in ("RA_Attack","Combined_Attack") else ""
        print(f"  {cls:12s}  {precs[i]*100:>9.2f}%  {recs[i]*100:>7.2f}%  "
              f"{f1s[i]*100:>7.2f}%{flag}")
    mf1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
    print(f"  {'macro':12s}  {'—':>10}  {'—':>8}  {mf1*100:>7.2f}%")
    return f1s, precs, recs


# ─────────────────────────────────────────────────────────────────────────────
# STEP 0 — Load nodes.csv
# ─────────────────────────────────────────────────────────────────────────────
print("=" * 70)
print("STEP 0 — Loading node identity (nodes.csv) ...")
print("=" * 70)
if not Path(NODES_CSV_PATH).exists():
    print(f"\n  [FATAL] nodes.csv not found: {NODES_CSV_PATH}"); sys.exit(1)
ATTACKER_MAC = None; ROUTER_MAC = None; VICTIM_MACS = set()
with open(NODES_CSV_PATH) as f:
    for row in csv.DictReader(f):
        role = row["role"].strip().lower(); mac = row["mac"].strip().lower()
        if   role == "attacker": ATTACKER_MAC = mac
        elif role == "router":   ROUTER_MAC   = mac
        elif role == "victim":   VICTIM_MACS.add(mac)
if not ATTACKER_MAC: print("  [FATAL] Attacker missing."); sys.exit(1)
if not ROUTER_MAC:   print("  [FATAL] Router missing.");   sys.exit(1)
print(f"  Attacker MAC : {ATTACKER_MAC}")
print(f"  Router   MAC : {ROUTER_MAC}")
print(f"  Victim   MACs: {len(VICTIM_MACS)}")
for m in sorted(VICTIM_MACS): print(f"    {m}")
print("\n  NOTE: Attacker MAC spoofed — events.csv is sole label source.")


# ─────────────────────────────────────────────────────────────────────────────
# STEP 1 — Load events.csv
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 70)
print("STEP 1 — Loading events.csv and building phase timeline ...")
print("=" * 70)
phase_timeline = []; events_loaded = False
if Path(EVENTS_CSV_PATH).exists():
    all_phase_names = set()
    with open(EVENTS_CSV_PATH) as f:
        for row in csv.DictReader(f): all_phase_names.add(row["phase"].strip())
    ATTACK_PHASE_MAP = {p: phase_name_to_label(p) for p in all_phase_names}
    print("  Phase name → label mapping:")
    for p in sorted(all_phase_names): print(f"    {p:30s} → {ATTACK_PHASE_MAP[p]}")
    phase_starts = {}
    with open(EVENTS_CSV_PATH) as f:
        for row in csv.DictReader(f):
            evt = row["event"].strip(); phase = row["phase"].strip()
            ts  = float(row["ts_epoch"])
            if evt == "phase_start": phase_starts[phase] = ts
            elif evt == "phase_end" and phase in phase_starts:
                phase_timeline.append((phase_starts[phase], ts,
                    ATTACK_PHASE_MAP.get(phase, "Normal"), phase))
                del phase_starts[phase]
    for phase, ts_s in phase_starts.items():
        phase_timeline.append((ts_s, float("inf"),
            ATTACK_PHASE_MAP.get(phase, "Normal"), phase))
    phase_timeline.sort(key=lambda x: x[0]); events_loaded = True
    all_ends = [t[1] for t in phase_timeline if t[1] != float("inf")]
    if all_ends:
        capture_start = min(t[0] for t in phase_timeline)
        capture_end = max(all_ends); total_dur = capture_end - capture_start
        attack_dur = sum(min(e, capture_end) - s
                         for s, e, lbl, _ in phase_timeline if lbl != "Normal")
        print(f"\n  Attack: {attack_dur:.0f}s / {total_dur:.0f}s "
              f"({100*attack_dur/total_dur:.1f}%)")
    print(f"\n  Phase timeline ({len(phase_timeline)} intervals):")
    for s, e, lbl, pname in phase_timeline:
        e_str = f"{e:.1f}" if e != float("inf") else "∞"
        print(f"    {s:.1f} → {e_str}  [{lbl:16s}]  ({pname})")

def timeline_label(ts):
    for start, end, label, _ in phase_timeline:
        if start <= ts < end: return label
    return "Normal"


# ─────────────────────────────────────────────────────────────────────────────
# STEP 2 — Parse PCAP
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 70)
print("STEP 2 — Parsing PCAP ...")
print("=" * 70)
if not Path(PCAP_PATH).exists():
    print(f"\n  [FATAL] PCAP not found: {PCAP_PATH}"); sys.exit(1)
pkt_info_list = []; ra_times_src = defaultdict(list); ns_times_src = defaultdict(list)
all_ra_ts = []; all_ns_ts = []; total_pkts = 0; ipv6_pkts = 0; no_ether_warn = 0
with PcapReader(PCAP_PATH) as pr:
    for pkt in pr:
        total_pkts += 1
        if not pkt.haslayer(IPv6): continue
        ipv6_pkts += 1
        ts = float(pkt.time); src = str(pkt[IPv6].src); dst = str(pkt[IPv6].dst)
        if pkt.haslayer(Ether): eth_src = pkt[Ether].src.lower()
        else: eth_src = ""; no_ether_warn += 1
        is_ra = pkt.haslayer(ICMPv6ND_RA); is_ns = pkt.haslayer(ICMPv6ND_NS)
        is_na = pkt.haslayer(ICMPv6ND_NA)
        is_echo = pkt.haslayer(ICMPv6EchoRequest) or pkt.haslayer(ICMPv6EchoReply)
        if is_ra: ra_times_src[src].append(ts); all_ra_ts.append(ts)
        if is_ns: ns_times_src[src].append(ts); all_ns_ts.append(ts)
        ns_target = str(getattr(pkt[ICMPv6ND_NS], "tgt", "::")) if is_ns else ""
        pkt_info_list.append({"ts": ts, "src": src, "dst": dst, "eth_src": eth_src,
            "pkt_len": len(pkt), "is_ra": int(is_ra), "is_ns": int(is_ns),
            "is_na": int(is_na), "is_echo": int(is_echo), "ns_target": ns_target})
print(f"\n  Total packets : {total_pkts:,}")
print(f"  IPv6 packets  : {ipv6_pkts:,}")
print(f"  RA packets    : {len(all_ra_ts):,}")
print(f"  NS packets    : {len(all_ns_ts):,}")


# ─────────────────────────────────────────────────────────────────────────────
# STEPS 3-4 — Rate-based detection (diagnostic)
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 70)
print("STEP 3-4 — Rate-based detection (diagnostic) ...")
print("=" * 70)
ra_global_bucket = Counter(int(t) for t in all_ra_ts)
ns_global_bucket = Counter(int(t) for t in all_ns_ts)
attack_ra_wins = {b for b, c in ra_global_bucket.items() if c >= RA_GLOBAL_THRESHOLD}
attack_ns_wins = {b for b, c in ns_global_bucket.items() if c >= NS_GLOBAL_THRESHOLD}
print(f"  RA attack windows (≥{RA_GLOBAL_THRESHOLD} pkt/s): {len(attack_ra_wins)}")
print(f"  NS attack windows (≥{NS_GLOBAL_THRESHOLD} pkt/s): {len(attack_ns_wins)}")


# ─────────────────────────────────────────────────────────────────────────────
# STEP 5 — Per-packet labeling
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 70)
print("STEP 5 — Per-packet labeling ...")
print("=" * 70)
pkt_rows = []
for info in pkt_info_list:
    ts = info["ts"]; eth_src = info["eth_src"]
    label = timeline_label(ts) if events_loaded else (
        "RA_Attack" if eth_src == ATTACKER_MAC and info["is_ra"] else
        "ND_Attack" if eth_src == ATTACKER_MAC else "Normal")
    mac_label = ("RA_Attack" if eth_src == ATTACKER_MAC and info["is_ra"] else
                 "ND_Attack" if eth_src == ATTACKER_MAC else
                 "Normal"    if eth_src == ROUTER_MAC else
                 timeline_label(ts) if eth_src in VICTIM_MACS else "Normal")
    pkt_rows.append({**info, "mac_label": mac_label, "label": label})
df_pkts = pd.DataFrame(pkt_rows)
print(f"  Per-packet DataFrame : {df_pkts.shape}")
vc_pkts = df_pkts["label"].value_counts()
print("\n  Per-packet class distribution:")
for cls in ["RA_Attack","ND_Attack","Combined_Attack","Normal"]:
    n = vc_pkts.get(cls, 0); pct = 100*n/len(df_pkts) if len(df_pkts) else 0
    print(f"    {cls:16s}: {n:8,d}  ({pct:.2f}%)")
if not any(vc_pkts.get(c, 0) for c in ["RA_Attack","ND_Attack","Combined_Attack"]):
    print("\n  [FATAL] Zero attack packets."); sys.exit(1)
df_pkts.to_csv(CSV_PKT_PATH, index=False)
print(f"\n  Per-packet CSV → {CSV_PKT_PATH}")


# ─────────────────────────────────────────────────────────────────────────────
# STEP 6 — Window feature extraction
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 70)
print("STEP 6 — Window-based feature extraction ...")
print("=" * 70)
buckets = defaultdict(list)
for info in pkt_info_list: buckets[int(info["ts"])].append(info)
window_rows = []; sorted_buckets = sorted(buckets.keys()); bkt_rate_hist = {}

for b in sorted_buckets:
    pkts = buckets[b]; n = len(pkts); bkt_rate_hist[b] = n
    ts_arr = sorted(p["ts"] for p in pkts)
    if n >= 2:
        iats = [max((ts_arr[i+1]-ts_arr[i])*1000.0, 1e-6) for i in range(n-1)]
        m_iat = float(np.mean(iats)); s_iat = float(np.std(iats)); mi_iat = float(np.min(iats))
    else: m_iat = s_iat = mi_iat = 0.0
    burstiness = s_iat / (m_iat + 1e-6)
    ra_cnt = sum(p["is_ra"] for p in pkts); ns_cnt = sum(p["is_ns"] for p in pkts)
    na_cnt = sum(p["is_na"] for p in pkts)
    ra_burst_rate = max_burst_rate(sorted(p["ts"] for p in pkts if p["is_ra"]), BURST_SUBWINDOW)
    ns_burst_rate = max_burst_rate(sorted(p["ts"] for p in pkts if p["is_ns"]), BURST_SUBWINDOW)
    src_set = {p["src"] for p in pkts}; dst_set = {p["dst"] for p in pkts}
    mc_cnt = sum(1 for p in pkts if p["dst"].startswith("ff"))
    ra_src_set = {p["src"] for p in pkts if p["is_ra"]}
    ns_tgt_set = {p["ns_target"] for p in pkts if p["is_ns"] and p["ns_target"]}
    multicast_ratio = mc_cnt/n; src_diversity = len(src_set)/n
    ra_src_div = len(ra_src_set)/max(ra_cnt,1); ns_unique_tgt_r = len(ns_tgt_set)/max(ns_cnt,1)
    s3t = sum(bkt_rate_hist.get(b-i,0) for i in range(1,4))
    s3nd = sum(sum(p["is_ra"]+p["is_ns"]+p["is_na"] for p in buckets.get(b-i,[]))
               for i in range(1,4))
    sliding_nd_ratio_3s = s3nd/max(s3t,1)
    label = timeline_label(float(b)) if events_loaded else "Normal"
    has_atk_ra = any(p["eth_src"]==ATTACKER_MAC and p["is_ra"] for p in pkts)
    has_atk_ns = any(p["eth_src"]==ATTACKER_MAC and p["is_ns"] for p in pkts)
    mac_win_label = "RA_Attack" if has_atk_ra else "ND_Attack" if has_atk_ns else "Normal"
    window_rows.append({
        "time_bucket": b, "pkt_rate": float(n),
        "ra_rate": float(ra_cnt), "ns_rate": float(ns_cnt), "na_rate": float(na_cnt),
        "ra_burst_rate": ra_burst_rate, "ns_burst_rate": ns_burst_rate,
        "unique_src_count": float(len(src_set)), "unique_dst_count": float(len(dst_set)),
        "mean_iat_ms": m_iat, "std_iat_ms": s_iat, "min_iat_ms": mi_iat,
        "burstiness": burstiness, "multicast_ratio": multicast_ratio,
        "src_diversity": src_diversity, "ra_src_diversity": ra_src_div,
        "ns_unique_tgt_r": ns_unique_tgt_r, "sliding_nd_ratio_3s": sliding_nd_ratio_3s,
        "label": label, "mac_label": mac_win_label,
    })
df_win = pd.DataFrame(window_rows)
df_win.to_csv(CSV_WIN_PATH, index=False)
print(f"  Window DataFrame shape : {df_win.shape}")
print(f"  Time range : {sorted_buckets[0]} → {sorted_buckets[-1]}")
vc_win = df_win["label"].value_counts()
print("\n  Window class distribution:")
for cls in ["RA_Attack","ND_Attack","Combined_Attack","Normal"]:
    n = vc_win.get(cls,0); pct = 100*n/len(df_win) if len(df_win) else 0
    print(f"    {cls:16s}: {n:5,d} windows  ({pct:.1f}%)")


# ─────────────────────────────────────────────────────────────────────────────
# STEP 7 — Class balance assertion
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 70)
print("STEP 7 — Class balance assertion ...")
print("=" * 70)
class_counts = {cls: int(vc_win.get(cls,0))
                for cls in ["Normal","RA_Attack","ND_Attack","Combined_Attack"]}
missing = [c for c,n in class_counts.items() if n < MIN_CLASS_WINDOWS]
for cls, n in class_counts.items():
    status = f"OK ({n:,})" if n >= MIN_CLASS_WINDOWS else f"FAIL — only {n}"
    print(f"    {cls:16s}: {status}")
if missing: print(f"\n  [FATAL] Insufficient: {missing}"); sys.exit(1)
total_w = sum(class_counts.values())
max_ratio = max(class_counts.values()) / max(min(class_counts.values()), 1)
print(f"\n  Total windows   : {total_w}")
print(f"  Imbalance ratio : {max_ratio:.1f}x")


# ─────────────────────────────────────────────────────────────────────────────
# STEP 8 — ML preprocessing + STRATIFIED splits (NO temporal/chronological)
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 70)
print("STEP 8 — ML preprocessing (Stratified Splits) ...")
print("=" * 70)

_exclude_meta = {"time_bucket","label","mac_label"} | ALWAYS_EXCLUDED_FEATURES
print(f"\n  Always-excluded: {sorted(ALWAYS_EXCLUDED_FEATURES)}")
FEATURE_COLS = [c for c in df_win.columns if c not in _exclude_meta]
non_const = df_win[FEATURE_COLS].nunique() > 1
FEATURE_COLS = [c for c in FEATURE_COLS if non_const[c]]
print(f"  Feature pool ({len(FEATURE_COLS)}): {FEATURE_COLS}")

X_raw = df_win[FEATURE_COLS].values.astype(float)
le = LabelEncoder()
y  = le.fit_transform(df_win["label"])
n_cls = len(le.classes_)
print(f"\n  Classes      : {list(le.classes_)}")
print(f"  Total windows: {len(y):,}")

# ── Standard scaling ─────────────────────────────────────────────────────────
scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

# ── Split A — Stratified 80/20 (random_state=42) ─────────────────────────────
X_train_raw, X_test, y_train_raw, y_test = train_test_split(
    X_scaled, y,
    test_size=TEST_SIZE,
    stratify=y,
    random_state=RANDOM_STATE
)

print(f"\n  Split A — Stratified 80/20 (random_state={RANDOM_STATE}) BEFORE SMOTE:")
print(f"    Train: {len(y_train_raw):,}   Test: {len(y_test):,}")
for ci, cn in enumerate(le.classes_):
    tr_n = int(np.sum(y_train_raw == ci))
    te_n = int(np.sum(y_test     == ci))
    print(f"    {cn:16s}  train={tr_n:,}  test={te_n:,}")

# ── SMOTE on training data only ───────────────────────────────────────────────
majority_n         = max(Counter(y_train_raw).values())
minority_threshold = int(majority_n * 0.15)
needs_smote        = any(v < minority_threshold for v in Counter(y_train_raw).values())

if _HAS_SMOTE and needs_smote:
    min_class_n = min(Counter(y_train_raw).values())
    smote_k     = max(1, min(5, min_class_n - 1))
    sm          = SMOTE(random_state=RANDOM_STATE, k_neighbors=smote_k)
    X_train, y_train = sm.fit_resample(X_train_raw, y_train_raw)
    print(f"\n  [SMOTE] Applied (k_neighbors={smote_k}):")
    print(f"    Before: {len(y_train_raw):,}  →  After: {len(y_train):,}")
    for ci, cn in enumerate(le.classes_):
        before = int(np.sum(y_train_raw == ci))
        after  = int(np.sum(y_train     == ci))
        print(f"    {cn:16s}  {before:,} → {after:,}  (+{after - before:,})")
else:
    X_train, y_train = X_train_raw, y_train_raw
    if not _HAS_SMOTE:
        print("\n  [SMOTE] Not available — using original training data.")
    else:
        print("\n  [SMOTE] Classes balanced enough — skipped.")

# ── Split B — Stratified 80/20 (random_state=123, different partition) ────────
# Used in Step 12 and Step 16 to assess variance across random partitions
X_tr_b_raw, X_te_b, y_tr_b_raw, y_te_b = train_test_split(
    X_scaled, y,
    test_size=TEST_SIZE,
    stratify=y,
    random_state=RANDOM_STATE_B
)

if _HAS_SMOTE and needs_smote:
    min_b = min(Counter(y_tr_b_raw).values())
    if min_b >= 2:
        smote_k_b     = max(1, min(5, min_b - 1))
        sm_b          = SMOTE(random_state=RANDOM_STATE_B, k_neighbors=smote_k_b)
        X_tr_b, y_tr_b = sm_b.fit_resample(X_tr_b_raw, y_tr_b_raw)
        print(f"\n  Split B SMOTE: {len(y_tr_b_raw):,} → {len(y_tr_b):,}")
    else:
        X_tr_b, y_tr_b = X_tr_b_raw, y_tr_b_raw
else:
    X_tr_b, y_tr_b = X_tr_b_raw, y_tr_b_raw

print(f"\n  Split B — Stratified 80/20 (random_state={RANDOM_STATE_B}):")
print(f"    Train: {len(y_tr_b):,}   Test: {len(y_te_b):,}")
for ci, cn in enumerate(le.classes_):
    print(f"    {cn:16s}  train={int(np.sum(y_tr_b == ci)):,}  "
          f"test={int(np.sum(y_te_b == ci)):,}")

# ── Aliases for downstream compatibility ─────────────────────────────────────
# These replace the old temporal split variables throughout Steps 9–16
X_tr_temp  = X_tr_b
y_tr_temp  = y_tr_b
X_te_temp  = X_te_b
y_te_temp  = y_te_b


# ─────────────────────────────────────────────────────────────────────────────
# STEP 8b — ANOVA-F feature ranking (train-only)
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 70)
print("STEP 8b — SelectKBest feature ranking (ANOVA-F, train-only) ...")
print("=" * 70)
skb_all = SelectKBest(score_func=f_classif, k="all")
skb_all.fit(X_train, y_train)
skb_scores  = skb_all.scores_
skb_pvalues = skb_all.pvalues_
skb_ranking         = np.argsort(skb_scores)[::-1]
skb_ranked_features = [FEATURE_COLS[i] for i in skb_ranking]
skb_ranked_scores   = [skb_scores[i]   for i in skb_ranking]
skb_ranked_pvals    = [skb_pvalues[i]  for i in skb_ranking]

print(f"\n  ANOVA-F ranking (n_train={len(y_train):,}):\n")
print(f"  {'Rank':>5}  {'Feature':28s}  {'F-score':>10}  {'p-value':>12}")
print(f"  {'-'*5}  {'-'*28}  {'-'*10}  {'-'*12}")
for rank, (feat, fs, pv) in enumerate(
        zip(skb_ranked_features, skb_ranked_scores, skb_ranked_pvals), 1):
    mark = "★" if rank <= PRIMARY_K else " "
    print(f"  {rank:>5}{mark} {feat:28s}  {fs:>10.3f}  {pv:>12.2e}")
TOP_K_FEATURES = skb_ranked_features[:min(PRIMARY_K, len(FEATURE_COLS))]
TOP_K_IDX      = [FEATURE_COLS.index(f) for f in TOP_K_FEATURES]
print(f"\n  Primary model uses top-{len(TOP_K_FEATURES)}: {TOP_K_FEATURES}")


# ─────────────────────────────────────────────────────────────────────────────
# STEP 9 — Single-feature leakage scan
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 70)
print("STEP 9 — Single-feature leakage scan ...")
print("=" * 70)
print(f"  Threshold: ROC-AUC > {LEAKAGE_AUC_WARN} → warn\n")
print(f"  {'Feature':30s}  {'ROC-AUC':>8}  {'BalAcc':>8}  Status")
print(f"  {'-'*30}  {'-'*8}  {'-'*8}  ------")
leaky_features = []; high_solo_features = []

for fi, feat in enumerate(FEATURE_COLS):
    # Use Split A train/test for leakage scan
    x1d_tr = X_train[:, fi].reshape(-1, 1)
    x1d_te = X_test[:,  fi].reshape(-1, 1)
    clf1 = make_rf(n_estimators=50, max_depth=3)
    clf1.fit(x1d_tr, y_train)
    prob1 = safe_predict_proba(clf1, x1d_te, n_cls)
    pred1 = clf1.predict(x1d_te)
    auc1, _ = safe_roc_auc(y_test, prob1, n_cls)
    bacc1   = balanced_accuracy_score(y_test, pred1)
    flags = []
    if not math.isnan(auc1) and auc1 > LEAKAGE_AUC_WARN:
        leaky_features.append(feat); flags.append("AUC-WARN")
    if bacc1 > MAX_SINGLE_FEAT_BACC:
        high_solo_features.append(feat); flags.append("BACC-EXCLUDED")
    flag_str = " ← " + " | ".join(flags) if flags else ""
    auc_str  = f"{auc1:>8.4f}" if not math.isnan(auc1) else "     NaN"
    print(f"  {feat:30s}  {auc_str}  {bacc1*100:>7.2f}%{flag_str}")

if leaky_features:
    print(f"\n  [WARN] High single-feature AUC: {leaky_features}")
    if events_loaded: print("  Labels from events.csv — genuine signal.")
else:
    print(f"\n  [OK] No single feature exceeds AUC {LEAKAGE_AUC_WARN}.")

if high_solo_features:
    print(f"  [BACC-CAP] Excluding: {high_solo_features}")
    keep_mask  = [c not in high_solo_features for c in FEATURE_COLS]
    FEATURE_COLS = [c for c, k in zip(FEATURE_COLS, keep_mask) if k]
    keep_idx   = [i for i, k in enumerate(keep_mask) if k]
    X_train    = X_train[:,   keep_idx]
    X_test     = X_test[:,    keep_idx]
    X_tr_temp  = X_tr_temp[:, keep_idx]
    X_te_temp  = X_te_temp[:, keep_idx]
    X_scaled   = X_scaled[:,  keep_idx]
    TOP_K_FEATURES      = [f for f in TOP_K_FEATURES      if f in FEATURE_COLS]
    TOP_K_IDX           = [FEATURE_COLS.index(f) for f in TOP_K_FEATURES]
    skb_ranked_features = [f for f in skb_ranked_features if f in FEATURE_COLS]


# ─────────────────────────────────────────────────────────────────────────────
# STEP 10 — XGBoost Hyperparameter Sweep (StratifiedKFold)
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 70)
print("STEP 10 — XGBoost Hyperparameter Sweep ...")
print("=" * 70)

if not _HAS_XGB:
    print("  [SKIP] xgboost not installed.")
    best_xgb_lr = 0.1; best_xgb_depth = 4; clf_xgb_final = None
else:
    print(f"\n  Top-{len(TOP_K_FEATURES)}: {TOP_K_FEATURES}")
    print(f"  LR ∈ {XGB_LR_SWEEP} × depth ∈ {XGB_DEPTH_SWEEP}")
    print(f"  gamma=1.0, reg_lambda=2.0")
    X_tr_k = X_train[:, TOP_K_IDX]
    X_te_k = X_test[:,  TOP_K_IDX]
    skf_xgb = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    print(f"  StratifiedKFold(n_splits={N_FOLDS})\n")
    xgb_results = []
    print(f"  {'LR':>5}  {'Depth':>5}  {'CV-BalAcc':>10}  {'Test-BalAcc':>11}  "
          f"{'MacroF1':>8}  {'TrainAcc':>9}")
    print(f"  {'-'*5}  {'-'*5}  {'-'*10}  {'-'*11}  {'-'*8}  {'-'*9}")
    for lr in XGB_LR_SWEEP:
        for depth in XGB_DEPTH_SWEEP:
            cv_baccs = []
            for tr_i, val_i in skf_xgb.split(X_tr_k, y_train):
                clf_cv = make_xgb(lr, depth)
                clf_cv.fit(X_tr_k[tr_i], y_train[tr_i])
                cv_baccs.append(balanced_accuracy_score(
                    y_train[val_i], clf_cv.predict(X_tr_k[val_i])))
            cv_ba    = float(np.mean(cv_baccs))
            clf_full = make_xgb(lr, depth)
            clf_full.fit(X_tr_k, y_train)
            pred_te  = clf_full.predict(X_te_k)
            pred_tr  = clf_full.predict(X_tr_k)
            ba       = balanced_accuracy_score(y_test, pred_te)
            mf1      = f1_score(y_test, pred_te, average="macro", zero_division=0)
            tra      = accuracy_score(y_train, pred_tr)
            xgb_results.append({"lr": lr, "depth": depth, "cv_bacc": cv_ba,
                "test_bacc": ba, "mf1": mf1, "train_acc": tra, "model": clf_full})
            print(f"  {lr:>5.2f}  {depth:>5}  {cv_ba*100:>9.2f}%  "
                  f"{ba*100:>10.2f}%  {mf1*100:>7.2f}%  {tra*100:>8.2f}%")
    best_xgb_idx   = max(range(len(xgb_results)), key=lambda i: xgb_results[i]["cv_bacc"])
    best_xgb       = xgb_results[best_xgb_idx]
    best_xgb_lr    = best_xgb["lr"]
    best_xgb_depth = best_xgb["depth"]
    clf_xgb_final  = best_xgb["model"]
    print(f"\n  ★ Best XGBoost: LR={best_xgb_lr}, depth={best_xgb_depth}  "
          f"(CV BalAcc={best_xgb['cv_bacc']*100:.2f}%)")


# ─────────────────────────────────────────────────────────────────────────────
# STEP 10a — Random Forest Sweep
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 70)
print("STEP 10a — Random Forest Hyperparameter Sweep ...")
print("=" * 70)
print(f"\n  Grid: n_estimators ∈ {RF_NEST_SWEEP} × depth ∈ {RF_DEPTH_SWEEP}")
X_tr_k = X_train[:, TOP_K_IDX]
X_te_k = X_test[:,  TOP_K_IDX]
skf_rf  = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
rf_results = []
print(f"\n  {'N_est':>5}  {'Depth':>5}  {'CV-BalAcc':>10}  {'Test-BalAcc':>11}  "
      f"{'MacroF1':>8}  {'TrainAcc':>9}")
print(f"  {'-'*5}  {'-'*5}  {'-'*10}  {'-'*11}  {'-'*8}  {'-'*9}")
for n_est in RF_NEST_SWEEP:
    for depth in RF_DEPTH_SWEEP:
        cv_baccs = []
        for tr_i, val_i in skf_rf.split(X_tr_k, y_train):
            clf_cv = make_rf(n_est, depth)
            clf_cv.fit(X_tr_k[tr_i], y_train[tr_i])
            cv_baccs.append(balanced_accuracy_score(
                y_train[val_i], clf_cv.predict(X_tr_k[val_i])))
        cv_ba    = float(np.mean(cv_baccs))
        clf_full = make_rf(n_est, depth)
        clf_full.fit(X_tr_k, y_train)
        pred_te  = clf_full.predict(X_te_k)
        pred_tr  = clf_full.predict(X_tr_k)
        ba       = balanced_accuracy_score(y_test, pred_te)
        mf1      = f1_score(y_test, pred_te, average="macro", zero_division=0)
        tra      = accuracy_score(y_train, pred_tr)
        d_str    = str(depth) if depth is not None else "None"
        rf_results.append({"n_est": n_est, "depth": depth, "cv_bacc": cv_ba,
            "test_bacc": ba, "mf1": mf1, "train_acc": tra, "model": clf_full})
        print(f"  {n_est:>5}  {d_str:>5}  {cv_ba*100:>9.2f}%  "
              f"{ba*100:>10.2f}%  {mf1*100:>7.2f}%  {tra*100:>8.2f}%")
best_rf_idx   = max(range(len(rf_results)), key=lambda i: rf_results[i]["cv_bacc"])
best_rf       = rf_results[best_rf_idx]
best_rf_nest  = best_rf["n_est"]
best_rf_depth = best_rf["depth"]
clf_rf_final  = best_rf["model"]
d_str = str(best_rf_depth) if best_rf_depth is not None else "None"
print(f"\n  ★ Best RF: n_estimators={best_rf_nest}, depth={d_str}  "
      f"(CV BalAcc={best_rf['cv_bacc']*100:.2f}%)")


# ─────────────────────────────────────────────────────────────────────────────
# STEP 10b — Feature-count sensitivity
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 70)
print(f"STEP 10b — Feature-count sensitivity  k ∈ {K_FEAT_SWEEP} ...")
print("=" * 70)
_k_sweep_eff    = [k for k in K_FEAT_SWEEP if k <= len(FEATURE_COLS)] or [len(FEATURE_COLS)]
_anova_ranked   = skb_ranked_features
skf_kf          = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
kfeat_xgb = {}; kfeat_rf = {}

for model_label, make_fn, result_store in [
    ("XGBoost", lambda: make_xgb(best_xgb_lr, best_xgb_depth) if _HAS_XGB else None, kfeat_xgb),
    ("RF",      lambda: make_rf(best_rf_nest, best_rf_depth), kfeat_rf),
]:
    if make_fn() is None: print(f"\n  [SKIP] {model_label}"); continue
    print(f"\n  ── {model_label} ──")
    print(f"  {'k':>4}  {'Top features':38s}  {'CV-BalAcc':>10}  "
          f"{'Test-BalAcc':>11}  {'MacroF1':>8}")
    print(f"  {'-'*4}  {'-'*38}  {'-'*10}  {'-'*11}  {'-'*8}")
    for _k in _k_sweep_eff:
        _top_k = [f for f in _anova_ranked if f in FEATURE_COLS][:_k]
        _idx   = [FEATURE_COLS.index(f) for f in _top_k]
        _Xtr   = X_train[:, _idx]
        _Xte   = X_test[:,  _idx]
        _cv_baccs = []
        for _tri, _vali in skf_kf.split(_Xtr, y_train):
            _clf = make_fn(); _clf.fit(_Xtr[_tri], y_train[_tri])
            _cv_baccs.append(balanced_accuracy_score(
                y_train[_vali], _clf.predict(_Xtr[_vali])))
        _cv_ba = float(np.mean(_cv_baccs))
        _clf_k = make_fn(); _clf_k.fit(_Xtr, y_train)
        _pred  = _clf_k.predict(_Xte)
        _ba    = balanced_accuracy_score(y_test, _pred)
        _mf1   = f1_score(y_test, _pred, average="macro", zero_division=0)
        result_store[_k] = {"feats": _top_k, "cv_bacc": _cv_ba,
            "ba": _ba, "mf1": _mf1, "model": _clf_k, "idx": _idx}
        _preview = ", ".join(_top_k[:3]) + (f" +{_k-3}" if _k > 3 else "")
        print(f"  {_k:>4}  {_preview:38s}  {_cv_ba*100:>9.2f}%  "
              f"{_ba*100:>10.2f}%  {_mf1*100:>7.2f}%")
    best_k = max(result_store, key=lambda k: result_store[k]["cv_bacc"])
    print(f"  ★ Best k={best_k} for {model_label} "
          f"(CV BalAcc={result_store[best_k]['cv_bacc']*100:.2f}%)")

best_k_xgb = max(kfeat_xgb, key=lambda k: kfeat_xgb[k]["cv_bacc"]) if kfeat_xgb else PRIMARY_K
best_k_rf  = max(kfeat_rf,  key=lambda k: kfeat_rf[k]["cv_bacc"])  if kfeat_rf  else PRIMARY_K
print(f"\n  Agreement: XGB best k={best_k_xgb}, RF best k={best_k_rf}")


# ─────────────────────────────────────────────────────────────────────────────
# STEP 11 — Final model evaluation (Split A — Stratified Holdout)
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 70)
print("STEP 11 — Final model evaluation (Split A — Stratified Holdout) ...")
print("=" * 70)
X_tr_k  = X_train[:,   TOP_K_IDX]
X_te_k  = X_test[:,    TOP_K_IDX]
X_tr_tk = X_tr_temp[:, TOP_K_IDX]   # Split B top-k subset
X_te_tk = X_te_temp[:, TOP_K_IDX]   # Split B top-k subset

final_models = {}
for mname, clf_f in [("XGBoost", clf_xgb_final), ("RF", clf_rf_final)]:
    if clf_f is None: continue
    pred_te = clf_f.predict(X_te_k)
    prob_te = safe_predict_proba(clf_f, X_te_k, n_cls)
    pred_tr = clf_f.predict(X_tr_k)
    acc_te  = accuracy_score(y_test, pred_te)
    acc_tr  = accuracy_score(y_train, pred_tr)
    bacc_te = balanced_accuracy_score(y_test, pred_te)
    mf1_te  = f1_score(y_test, pred_te, average="macro", zero_division=0)
    gap     = acc_tr - acc_te
    roc, roc_note = safe_roc_auc(y_test, prob_te, n_cls)
    pr_auc  = safe_pr_auc(y_test, prob_te, n_cls)
    final_models[mname] = {
        "clf": clf_f, "pred": pred_te, "prob": prob_te,
        "acc_tr": acc_tr, "acc_te": acc_te, "bacc": bacc_te,
        "mf1": mf1_te, "gap": gap, "roc": roc, "pr_auc": pr_auc
    }
    print(f"\n  ── {mname} ──")
    print(f"  Train Accuracy    : {acc_tr*100:.2f}%")
    print(f"  Test  Accuracy    : {acc_te*100:.2f}%")
    print(f"  Overfitting gap   : {gap*100:+.2f}%  "
          f"{'[WARN]' if gap > OVERFIT_GAP_WARN else '[OK]'}")
    print(f"  Balanced Accuracy : {bacc_te*100:.2f}%")
    print(f"  Macro F1          : {mf1_te*100:.2f}%")
    print(f"  ROC-AUC           : {roc:.4f}  [{roc_note}]")
    print(f"  PR-AUC            : {pr_auc:.4f}")
    f1s_a, precs_a, recs_a = print_per_class_f1(
        y_test, pred_te, le, tag=f"[{mname} Split A]")
    final_models[mname].update({"f1s_a": f1s_a, "precs_a": precs_a, "recs_a": recs_a})


# ─────────────────────────────────────────────────────────────────────────────
# STEP 12 — Split B evaluation + 5-Fold CV stability
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 70)
print("STEP 12 — Split B (different stratified partition) + CV stability ...")
print("=" * 70)

temp_models = {}
for mname, factory in [
    ("XGBoost", lambda: make_xgb(best_xgb_lr, best_xgb_depth) if _HAS_XGB else None),
    ("RF",      lambda: make_rf(best_rf_nest, best_rf_depth)),
]:
    clf_t = factory()
    if clf_t is None: continue

    # ── Evaluate on Split B holdout ───────────────────────────────────────
    clf_t.fit(X_tr_tk, y_tr_temp)
    pred_t  = clf_t.predict(X_te_tk)
    prob_t  = safe_predict_proba(clf_t, X_te_tk, n_cls)
    bacc_t  = balanced_accuracy_score(y_te_temp, pred_t)
    mf1_t   = f1_score(y_te_temp, pred_t, average="macro", zero_division=0)
    roc_t, roc_note_t = safe_roc_auc(y_te_temp, prob_t, n_cls)
    pr_t    = safe_pr_auc(y_te_temp, prob_t, n_cls)

    # ── 5-Fold CV on full dataset ────────────────────────────────────────
    est_cv = factory()
    cv_res = cross_validate(
        est_cv,
        X_scaled[:, TOP_K_IDX], y,
        cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE),
        scoring={"bacc": "balanced_accuracy", "f1": "f1_macro"},
        return_train_score=True, n_jobs=1
    )
    cv_mean_bacc  = float(np.mean(cv_res["test_bacc"]))
    cv_std_bacc   = float(np.std( cv_res["test_bacc"]))
    cv_mean_f1    = float(np.mean(cv_res["test_f1"]))
    cv_mean_tr_ba = float(np.mean(cv_res["train_bacc"]))
    cv_gap        = cv_mean_tr_ba - cv_mean_bacc

    delta = final_models.get(mname, {}).get("bacc", bacc_t) - bacc_t

    temp_models[mname] = {
        "clf": clf_t, "pred": pred_t, "prob": prob_t,
        "bacc": bacc_t, "mf1": mf1_t, "roc": roc_t, "pr_auc": pr_t,
        "cv_mean_bacc": cv_mean_bacc, "cv_std_bacc": cv_std_bacc,
        "f1s_b":   f1_score(y_te_temp, pred_t, average=None,
                             labels=list(range(n_cls)), zero_division=0),
        "precs_b": precision_score(y_te_temp, pred_t, labels=list(range(n_cls)),
                                   average=None, zero_division=0),
        "recs_b":  recall_score(y_te_temp, pred_t, labels=list(range(n_cls)),
                                average=None, zero_division=0),
    }

    print(f"\n  ── {mname} (Split B — different stratified partition) ──")
    print(f"  Balanced Accuracy : {bacc_t*100:.2f}%")
    print(f"  Macro F1          : {mf1_t*100:.2f}%")
    print(f"  ROC-AUC           : {roc_t:.4f}  [{roc_note_t}]")
    print(f"  PR-AUC            : {pr_t:.4f}")
    print(f"  Variance (A vs B) : {delta*100:+.2f}%")
    print(f"\n  5-Fold CV (full dataset):")
    print(f"  CV BalAcc : {cv_mean_bacc*100:.2f}% ± {cv_std_bacc*100:.2f}%")
    print(f"  CV MacroF1: {cv_mean_f1*100:.2f}%")
    print(f"  CV gap    : {cv_gap*100:+.2f}%")
    for fi, fold_ba in enumerate(cv_res["test_bacc"]):
        print(f"    Fold {fi+1}: {fold_ba*100:.2f}%")
    print()
    f1s_b, precs_b, recs_b = print_per_class_f1(
        y_te_temp, pred_t, le, tag=f"[{mname} Split B]")
    temp_models[mname].update({"f1s_b": f1s_b, "precs_b": precs_b, "recs_b": recs_b})


# ─────────────────────────────────────────────────────────────────────────────
# STEPS 13-16 — Bootstrap CI, DummyClassifier, Learning curves, Comparison
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 70)
print(f"STEP 13 — Bootstrap 95% CI ({BOOTSTRAP_N_ITER} iterations) ...")
print("=" * 70)
_primary_name = "XGBoost" if "XGBoost" in final_models else "RF"
_ci_pred      = final_models[_primary_name]["pred"]
print(f"\n  Using {_primary_name} on Split A.")
ci_res = bootstrap_f1_ci(y_test, _ci_pred, list(le.classes_))
print(f"\n  {'Class':12s}  {'Mean F1':>8}  {'95% CI lo':>10}  {'95% CI hi':>10}")
print(f"  {'-'*12}  {'-'*8}  {'-'*10}  {'-'*10}")
for cls in list(le.classes_) + ["macro"]:
    m, lo, hi = ci_res[cls]
    print(f"  {cls:12s}  {m*100:>7.2f}%  {lo*100:>9.2f}%  {hi*100:>9.2f}%")

print("\n" + "=" * 70)
print("STEP 14 — DummyClassifier sanity check ...")
print("=" * 70)
dummy = DummyClassifier(strategy="stratified", random_state=RANDOM_STATE)
dummy.fit(X_tr_k, y_train)
pred_dummy  = dummy.predict(X_te_k)
bacc_dummy  = balanced_accuracy_score(y_test, pred_dummy)
for mname in final_models:
    bacc_real = final_models[mname]["bacc"]
    print(f"\n  {mname:10s}  BalAcc={bacc_real*100:.2f}%  "
          f"(Dummy={bacc_dummy*100:.2f}%, Δ={((bacc_real-bacc_dummy)*100):+.1f}pp)")
print(f"\n  [OK] All models outperform null model." if all(
    final_models[m]["bacc"] > bacc_dummy + 0.10 for m in final_models) else
    "\n  [WARN] Some models barely beat null.")

print("\n" + "=" * 70)
print("STEP 15 — Learning curves (StratifiedKFold) ...")
print("=" * 70)
n_classes_lc = len(np.unique(y_train))
lc_cv   = StratifiedKFold(n_splits=min(5, n_classes_lc*2), shuffle=False)
lc_data = {}
for mname, factory in [
    ("XGBoost", lambda: make_xgb(best_xgb_lr, best_xgb_depth) if _HAS_XGB else None),
    ("RF",      lambda: make_rf(best_rf_nest, best_rf_depth)),
]:
    est = factory()
    if est is None: continue
    train_sizes_abs, train_scores_lc, val_scores_lc = learning_curve(
        est, X_tr_k, y_train, cv=lc_cv,
        train_sizes=np.linspace(0.10, 1.0, 9),
        scoring="balanced_accuracy", n_jobs=1)
    train_mean = np.mean(train_scores_lc, axis=1)
    val_mean   = np.mean(val_scores_lc,   axis=1)
    lc_data[mname] = {"sizes": train_sizes_abs.tolist(),
                      "train": train_mean.tolist(), "val": val_mean.tolist()}
    print(f"\n  ── {mname} ──")
    for sz, tm, vm in zip(train_sizes_abs, train_mean, val_mean):
        print(f"  {int(sz):>8,}  Train={tm*100:.2f}%  Val={vm*100:.2f}%  "
              f"Gap={(tm-vm)*100:>+.2f}%")
    slope = val_mean[-1] - val_mean[-3]
    note  = "Val rising." if slope > 0.02 else ("Val declining." if slope < -0.02 else "Converged.")
    print(f"  [{note}]")

print("\n" + "=" * 70)
print("STEP 16 — Multi-model comparison (Split A vs Split B) ...")
print("=" * 70)

def _eval(clf, Xtr, ytr, Xte, yte):
    clf.fit(Xtr, ytr)
    pred = clf.predict(Xte)
    prob = safe_predict_proba(clf, Xte, n_cls)
    return {
        "pred": pred,
        "bacc": balanced_accuracy_score(yte, pred),
        "mf1":  f1_score(yte, pred, average="macro", zero_division=0),
        "f1s":  f1_score(yte, pred, average=None,
                         labels=list(range(n_cls)), zero_division=0),
        "roc":  safe_roc_auc(yte, prob, n_cls)[0],
    }

_model_defs = []
if _HAS_XGB:  _model_defs.append(("XGBoost",  lambda: make_xgb(best_xgb_lr, best_xgb_depth)))
_model_defs.append(             ("RF",         lambda: make_rf(best_rf_nest, best_rf_depth)))
if _HAS_LGB:  _model_defs.append(("LightGBM",  lambda: LGBMClassifier(
    n_estimators=100, max_depth=3, learning_rate=0.1,
    random_state=RANDOM_STATE, verbose=-1)))
if _HAS_CAT:  _model_defs.append(("CatBoost",  lambda: CatBoostClassifier(
    iterations=100, depth=3, learning_rate=0.1,
    random_state=RANDOM_STATE, verbose=0)))

comp_results = {}
print(f"\n  Features : {TOP_K_FEATURES}")
print(f"\n  Split A  : train={len(y_train):,} / test={len(y_test):,}  (random_state={RANDOM_STATE})")
print(f"  Split B  : train={len(y_tr_temp):,} / test={len(y_te_temp):,}  (random_state={RANDOM_STATE_B})")
print(f"  Comparison shows variance across two independent stratified partitions.\n")

hdr = (f"  {'Model':10s}  {'A-BalAcc':>9}  {'A-MacF1':>8}  "
       f"{'B-BalAcc':>9}  {'B-MacF1':>8}  {'Variance':>9}")
print(hdr)
print("  " + "-" * (len(hdr) - 2))

for mname, factory in _model_defs:
    # Split A: trained on X_tr_k / tested on X_te_k
    res_a = _eval(factory(), X_tr_k,  y_train,   X_te_k,  y_test)
    # Split B: trained on X_tr_tk / tested on X_te_tk  (different random partition)
    res_b = _eval(factory(), X_tr_tk, y_tr_temp, X_te_tk, y_te_temp)
    comp_results[mname] = {"A": res_a, "B": res_b}
    deg  = res_a["bacc"] - res_b["bacc"]
    flag = "  ← high variance" if abs(deg) > 0.10 else ""
    print(f"  {mname:10s}  {res_a['bacc']*100:>8.2f}%  {res_a['mf1']*100:>7.2f}%  "
          f"{res_b['bacc']*100:>8.2f}%  {res_b['mf1']*100:>7.2f}%  "
          f"{deg*100:>+8.2f}%{flag}")


# =============================================================================
# VISUALIZATIONS
# =============================================================================
print("\n" + "=" * 70)
print("PLOTS — generating 11 visualization files ...")
print("=" * 70)
cls_names   = list(le.classes_)
n_cls_plot  = len(cls_names)
cls_palette = [CLS_COLORS.get(c, ACCENT2) for c in cls_names]
all_cls_idx = list(range(n_cls_plot))

# ── PLOT 1 — accuracy_vs_features.png ────────────────────────────────────────
print("\nPLOT 1 — accuracy_vs_features.png ...")
fig, axes = plt.subplots(1, 2, figsize=(14, 6)); fig.patch.set_facecolor(DARK_BG)
fig.suptitle("Feature-count Sensitivity Sweep", color=TEXT_COL, fontsize=13)
for ax, (ml, kf) in zip(axes, [("XGBoost", kfeat_xgb), ("RF", kfeat_rf)]):
    style_ax(ax)
    if not kf:
        ax.text(0.5, 0.5, f"{ml}\nnot available", transform=ax.transAxes,
                ha="center", va="center", color=TEXT_COL)
        continue
    ks = sorted(kf.keys())
    ax.plot(ks, [kf[k]["cv_bacc"]*100 for k in ks], "-o",  color=ACCENT1, lw=2, ms=7, label="CV BalAcc")
    ax.plot(ks, [kf[k]["ba"]*100      for k in ks], "-s",  color=ACCENT2, lw=2, ms=7, label="Test BalAcc")
    ax.plot(ks, [kf[k]["mf1"]*100     for k in ks], "--^", color=ACCENT3, lw=2, ms=7, label="Test MacroF1")
    ax.set_xlabel("k (features)"); ax.set_ylabel("Score (%)")
    ax.set_title(f"{ml} — k-sweep", color=TEXT_COL)
    ax.set_xticks(ks); ax.set_ylim(0, 105); ax.legend(**LEG_KW)
plt.tight_layout(); savefig(fig, "accuracy_vs_features.png")

# ── PLOT 2 — selectkbest_ranking.png ─────────────────────────────────────────
print("PLOT 2 — selectkbest_ranking.png ...")
fig, ax = plt.subplots(figsize=(10, 7)); fig.patch.set_facecolor(DARK_BG); style_ax(ax)
_bar_cols = [ACCENT1 if f in TOP_K_FEATURES else ACCENT2 for f in skb_ranked_features]
ax.barh(skb_ranked_features[::-1], skb_ranked_scores[::-1],
        color=_bar_cols[::-1], alpha=0.88)
ax.set_xlabel("ANOVA-F Score")
ax.set_title(f"Feature Ranking  (★ red = top-{len(TOP_K_FEATURES)} used)", color=TEXT_COL)
ax.legend(handles=[mpatches.Patch(color=ACCENT1, label=f"Top-{len(TOP_K_FEATURES)} (used)"),
                   mpatches.Patch(color=ACCENT2, label="Not used")], **LEG_KW)
plt.tight_layout(); savefig(fig, "selectkbest_ranking.png")

# ── PLOT 3 — feature_analysis.png ────────────────────────────────────────────
print("PLOT 3 — feature_analysis.png ...")
fig, (ax_l, ax_r) = plt.subplots(1, 2, figsize=(16, 7)); fig.patch.set_facecolor(DARK_BG)
fig.suptitle("Feature Analysis", color=TEXT_COL, fontsize=13)
style_ax(ax_l)
_top_n  = min(10, len(skb_ranked_features))
_scores = np.array(skb_ranked_scores[:_top_n])
_rel    = _scores / _scores[0] * 100
_bcols  = [cls_palette[i % n_cls_plot] for i in range(_top_n)]
_bars   = ax_l.barh(skb_ranked_features[:_top_n][::-1], _rel[::-1],
                    color=_bcols[::-1], alpha=0.88)
for bar, val in zip(_bars, _rel[::-1]):
    ax_l.text(val+0.8, bar.get_y()+bar.get_height()/2,
              f"{val:.1f}%", va="center", color=TEXT_COL, fontsize=8)
ax_l.set_xlabel("Relative ANOVA-F Importance (%)")
ax_l.set_title("Top Feature Importances", color=TEXT_COL)
style_ax(ax_r)
_top8 = TOP_K_FEATURES[:min(8, len(TOP_K_FEATURES))]
_idx8 = [FEATURE_COLS.index(f) for f in _top8]
_corr_data = X_train_raw[:, _idx8]    # use original (non-SMOTE) training data
_corr = pd.DataFrame(_corr_data, columns=_top8).corr().values
_cmap = LinearSegmentedColormap.from_list("ids", [ACCENT3, PANEL_BG, ACCENT1], N=256)
_im   = ax_r.imshow(_corr, cmap=_cmap, vmin=-1, vmax=1, aspect="auto")
ax_r.set_xticks(range(len(_top8))); ax_r.set_yticks(range(len(_top8)))
ax_r.set_xticklabels(_top8, rotation=45, ha="right", color=TEXT_COL, fontsize=8)
ax_r.set_yticklabels(_top8, color=TEXT_COL, fontsize=8)
ax_r.set_title("Feature Correlation Heatmap (Top-8)", color=TEXT_COL)
for i in range(len(_top8)):
    for j in range(len(_top8)):
        ax_r.text(j, i, f"{_corr[i,j]:.2f}", ha="center", va="center",
                  color=TEXT_COL, fontsize=7)
cb = plt.colorbar(_im, ax=ax_r, fraction=0.046, pad=0.04)
cb.ax.yaxis.set_tick_params(color=TEXT_COL); cb.outline.set_edgecolor(BORDER)
plt.tight_layout(); savefig(fig, "feature_analysis.png")

# ── PLOT 4 — iat_per_class.png ───────────────────────────────────────────────
print("PLOT 4 — iat_per_class.png ...")
iat_per_class = []
for cls in cls_names:
    vals = df_win.loc[df_win["label"] == cls, "mean_iat_ms"].values
    vals = vals[vals > 0]
    iat_per_class.append(vals if len(vals) > 0 else np.array([0.001]))
fig, (ax_v, ax_b) = plt.subplots(1, 2, figsize=(14, 6)); fig.patch.set_facecolor(DARK_BG)
fig.suptitle("Inter-Arrival Time (IAT) Distribution per Class", color=TEXT_COL, fontsize=13)
style_ax(ax_v)
parts = ax_v.violinplot(iat_per_class, positions=range(n_cls_plot),
                         widths=0.6, showmedians=True, showextrema=False)
for i, pc in enumerate(parts["bodies"]):
    pc.set_facecolor(cls_palette[i]); pc.set_alpha(0.75)
parts["cmedians"].set_color(TEXT_COL); parts["cmedians"].set_linewidth(2)
ax_v.set_xticks(range(n_cls_plot))
ax_v.set_xticklabels(cls_names, color=TEXT_COL, fontsize=8, rotation=15)
ax_v.set_ylabel("Mean IAT (ms)"); ax_v.set_title("Violin", color=TEXT_COL)
ax_v.set_yscale("log")
style_ax(ax_b)
bp = ax_b.boxplot(iat_per_class, patch_artist=True, widths=0.5,
    medianprops=dict(color=TEXT_COL, lw=2),
    whiskerprops=dict(color=BORDER), capprops=dict(color=BORDER),
    flierprops=dict(markerfacecolor=BORDER, markersize=3))
for patch, col in zip(bp["boxes"], cls_palette):
    patch.set_facecolor(col); patch.set_alpha(0.80)
ax_b.set_xticks(range(1, n_cls_plot+1))
ax_b.set_xticklabels(cls_names, color=TEXT_COL, fontsize=8, rotation=15)
ax_b.set_ylabel("Mean IAT (ms)"); ax_b.set_title("Box-plot", color=TEXT_COL)
ax_b.set_yscale("log")
handles = [mpatches.Patch(color=c, label=n, alpha=0.85)
           for n, c in CLS_COLORS.items() if n in cls_names]
fig.legend(handles=handles, loc="upper right", bbox_to_anchor=(0.99, 0.99), **LEG_KW)
plt.tight_layout(); savefig(fig, "iat_per_class.png")

# ── PLOT 5 — learning_curve.png ──────────────────────────────────────────────
print("PLOT 5 — learning_curve.png ...")
lc_colors = {"XGBoost": (ACCENT1, ACCENT2), "RF": (ACCENT4, ACCENT5)}
fig, axes_lc = plt.subplots(1, max(len(lc_data), 1),
                             figsize=(7 * max(len(lc_data), 1), 6))
fig.patch.set_facecolor(DARK_BG)
fig.suptitle("Learning Curves — Balanced Accuracy", color=TEXT_COL, fontsize=13)
if len(lc_data) == 1: axes_lc = [axes_lc]
for ax, (mn, lcd) in zip(axes_lc, lc_data.items()):
    style_ax(ax)
    ct, cv_ = lc_colors.get(mn, (ACCENT1, ACCENT2))
    tr_a = np.array(lcd["train"]) * 100
    va_a = np.array(lcd["val"])   * 100
    sz   = lcd["sizes"]
    ax.plot(sz, tr_a, "-o", color=ct,  lw=2, ms=7, label="Train BalAcc")
    ax.plot(sz, va_a, "-s", color=cv_, lw=2, ms=7, label="Val BalAcc")
    ax.fill_between(sz, va_a, tr_a, alpha=0.12, color=ct, label="Overfit gap")
    ax.set_xlabel("Training Set Size"); ax.set_ylabel("Balanced Accuracy (%)")
    ax.set_title(mn, color=TEXT_COL); ax.set_ylim(30, 105)
    slope = va_a[-1] - va_a[-3] if len(va_a) >= 3 else 0
    note  = "↗ still rising" if slope > 2 else "✓ converged"
    ax.annotate(note, xy=(sz[-1], va_a[-1]),
                xytext=(sz[max(-3, -len(sz))], va_a[-1]+5),
                color=TEXT_COL, fontsize=9,
                arrowprops=dict(arrowstyle="->", color=TEXT_COL, lw=1.2))
    ax.legend(**LEG_KW)
plt.tight_layout(); savefig(fig, "learning_curve.png")

# # ── PLOT 6 — confusion_matrices.png ──────────────────────────────────────────
# print("PLOT 6 — confusion_matrices.png ...")
# cm_list = [(m, final_models[m]["pred"]) for m in final_models]
# n_cm    = len(cm_list)
# fig, axes_cm = plt.subplots(1, n_cm, figsize=(7*n_cm, 6))
# fig.patch.set_facecolor(DARK_BG)
# fig.suptitle("Confusion Matrices — Split A (Stratified 80/20)", color=TEXT_COL, fontsize=13)
# if n_cm == 1: axes_cm = [axes_cm]
# for ax, (mn, pred) in zip(axes_cm, cm_list):
#     ax.set_facecolor(PANEL_BG)
#     present = sorted(set(y_test) | set(pred))
#     names   = [le.classes_[i] for i in present]
#     cm_val  = confusion_matrix(y_test, pred, labels=present)
#     disp    = ConfusionMatrixDisplay(cm_val, display_labels=names)
#     disp.plot(ax=ax, colorbar=False, cmap="Blues")
#     ax.set_title(mn, color=TEXT_COL, fontsize=11)
#     ax.tick_params(colors=TEXT_COL)
#     ax.xaxis.label.set_color(TEXT_COL); ax.yaxis.label.set_color(TEXT_COL)
#     for txt in ax.texts: txt.set_color(TEXT_COL)
#     plt.setp(ax.get_xticklabels(), rotation=25, ha="right", fontsize=8)
# plt.tight_layout(); savefig(fig, "confusion_matrices.png")

# ── PLOT 6 — confusion_matrices.png ──────────────────────────────────────────
print("PLOT 6 — confusion_matrices.png ...")
cm_list = [(m, final_models[m]["pred"]) for m in final_models]
n_cm    = len(cm_list)
fig, axes_cm = plt.subplots(1, n_cm, figsize=(7*n_cm, 6))
fig.patch.set_facecolor(DARK_BG)
fig.suptitle("Confusion Matrices — Split A (Stratified 80/20)", color=TEXT_COL, fontsize=13)
if n_cm == 1: axes_cm = [axes_cm]

for ax, (mn, pred) in zip(axes_cm, cm_list):
    ax.set_facecolor(PANEL_BG)
    present = sorted(set(y_test) | set(pred))
    names   = [le.classes_[i] for i in present]
    cm_val  = confusion_matrix(y_test, pred, labels=present)

    # ── Plot with a high-contrast colormap ───────────────────────────────
    disp = ConfusionMatrixDisplay(cm_val, display_labels=names)
    disp.plot(ax=ax, colorbar=False, cmap="Blues")

    ax.set_title(mn, color=TEXT_COL, fontsize=11)
    ax.tick_params(colors=TEXT_COL)
    ax.xaxis.label.set_color(TEXT_COL)
    ax.yaxis.label.set_color(TEXT_COL)
    plt.setp(ax.get_xticklabels(), rotation=25, ha="right", fontsize=8, color=TEXT_COL)
    plt.setp(ax.get_yticklabels(), color=TEXT_COL)

    # ── Adaptive text contrast based on cell normalised intensity ─────────
    # Normalise cm values to [0,1] to judge brightness of the cell colour
    cm_norm = cm_val.astype(float) / (cm_val.max() + 1e-9)
    DARK_TEXT  = "#0d1117"   # near-black  — used on LIGHT cells (high value)
    LIGHT_TEXT = "#e6edf3"   # near-white  — used on DARK  cells (low value)
    THRESHOLD  = 0.45        # cells above this normalised value get dark text

    for text_obj in ax.texts:
        # text_obj position is in data coordinates matching the cell indices
        col_idx = int(round(text_obj.get_position()[0]))
        row_idx = int(round(text_obj.get_position()[1]))
        if 0 <= row_idx < cm_norm.shape[0] and 0 <= col_idx < cm_norm.shape[1]:
            brightness = cm_norm[row_idx, col_idx]
            text_obj.set_color(DARK_TEXT if brightness > THRESHOLD else LIGHT_TEXT)
        else:
            text_obj.set_color(LIGHT_TEXT)
        text_obj.set_fontsize(11)
        text_obj.set_fontweight("bold")

plt.tight_layout()
savefig(fig, "confusion_matrices.png")

# ── PLOT 7 — comparative_line_graphs.png ─────────────────────────────────────
print("PLOT 7 — comparative_line_graphs.png ...")
_mcols = {"XGBoost": ACCENT1, "RF": ACCENT2}
_mmrk  = {"XGBoost": "o",     "RF": "s"}
_kr    = sorted(_k_sweep_eff)
fig, axes_ln = plt.subplots(1, 3, figsize=(18, 6)); fig.patch.set_facecolor(DARK_BG)
fig.suptitle("Model Comparison — Metric vs Feature Count (k)", color=TEXT_COL, fontsize=13)
for ax, metric in zip(axes_ln, ["Test BalAcc", "Macro F1", "ROC-AUC"]):
    style_ax(ax)
    for mn, kf in [("XGBoost", kfeat_xgb), ("RF", kfeat_rf)]:
        if not kf: continue
        vals = []
        for _k in _kr:
            if _k in kf:
                vals.append(kf[_k]["ba"]*100 if metric == "Test BalAcc"
                            else kf[_k]["mf1"]*100)
            else:
                vals.append(comp_results.get(mn, {}).get("A", {}).get("bacc", 0)*100)
        ax.plot(_kr, vals, f"-{_mmrk.get(mn,'o')}",
                color=_mcols.get(mn, ACCENT2), lw=2.2, ms=8, label=mn)
        for xi, yi in zip(_kr, vals):
            ax.text(xi, yi+0.4, f"{yi:.1f}", ha="center",
                    color=_mcols.get(mn, ACCENT2), fontsize=7.5)
    ax.set_xlabel("k (features)"); ax.set_ylabel(f"{metric} (%)")
    ax.set_title(metric, color=TEXT_COL)
    ax.set_xticks(_kr); ax.set_ylim(50, 100); ax.legend(**LEG_KW)
plt.tight_layout(); savefig(fig, "comparative_line_graphs.png")

# ── PLOT 8 — comparative_bar_graphs.png ──────────────────────────────────────
print("PLOT 8 — comparative_bar_graphs.png ...")
_cmp = list(comp_results.keys()); _nc = len(_cmp); _bw = 0.18; _xc = np.arange(_nc)
fig, (ax_t, ax_b2) = plt.subplots(2, 1, figsize=(13, 11)); fig.patch.set_facecolor(DARK_BG)
fig.suptitle("Multi-model Comparison — Split A vs Split B (Stratified)",
             color=TEXT_COL, fontsize=13)
style_ax(ax_t)
_bm = [
    ("A BalAcc",  [comp_results[m]["A"]["bacc"]*100 for m in _cmp], ACCENT1),
    ("A MacroF1", [comp_results[m]["A"]["mf1"]*100  for m in _cmp], ACCENT4),
    ("B BalAcc",  [comp_results[m]["B"]["bacc"]*100 for m in _cmp], ACCENT2),
    ("B MacroF1", [comp_results[m]["B"]["mf1"]*100  for m in _cmp], ACCENT3),
]
_off = np.linspace(-(len(_bm)-1)*_bw/2, (len(_bm)-1)*_bw/2, len(_bm))
for (lbl, vals, col), off in zip(_bm, _off):
    rects = ax_t.bar(_xc + off, vals, _bw, label=lbl, color=col, alpha=0.88)
    for r in rects:
        ax_t.text(r.get_x()+r.get_width()/2, r.get_height()+0.3,
                  f"{r.get_height():.1f}", ha="center", color=TEXT_COL, fontsize=7.5)
ax_t.set_xticks(_xc); ax_t.set_xticklabels(_cmp, color=TEXT_COL)
ax_t.set_ylabel("Score (%)")
ax_t.set_title("Balanced Accuracy & Macro-F1 (Split A vs B)", color=TEXT_COL)
ax_t.set_ylim(50, 105); ax_t.legend(**LEG_KW, ncol=2)
style_ax(ax_b2); _bw2 = 0.35
_dba = [comp_results[m]["A"]["bacc"]*100 - comp_results[m]["B"]["bacc"]*100 for m in _cmp]
_dmf = [comp_results[m]["A"]["mf1"]*100  - comp_results[m]["B"]["mf1"]*100  for m in _cmp]
ax_b2.bar(_xc - _bw2/2, _dba, _bw2, label="ΔBalAcc (A−B)", color=ACCENT1, alpha=0.88)
ax_b2.bar(_xc + _bw2/2, _dmf, _bw2, label="ΔMacroF1 (A−B)", color=ACCENT4, alpha=0.88)
for xi, (vb, vm) in enumerate(zip(_dba, _dmf)):
    ax_b2.text(xi - _bw2/2, vb+0.1, f"{vb:.1f}", ha="center", color=TEXT_COL, fontsize=8)
    ax_b2.text(xi + _bw2/2, vm+0.1, f"{vm:.1f}", ha="center", color=TEXT_COL, fontsize=8)
ax_b2.axhline(5, color=ACCENT3, lw=1.4, ls="--", label="5 pp warn threshold")
ax_b2.set_xticks(_xc); ax_b2.set_xticklabels(_cmp, color=TEXT_COL)
ax_b2.set_ylabel("Partition Variance (pp)")
ax_b2.set_title("Partition Variance A → B", color=TEXT_COL)
ax_b2.legend(**LEG_KW)
plt.tight_layout(); savefig(fig, "comparative_bar_graphs.png")

# ── PLOT 9 — feature_mean_per_class.png ──────────────────────────────────────
print("PLOT 9 — feature_mean_per_class.png ...")
_top8_fm  = TOP_K_FEATURES[:min(8, len(TOP_K_FEATURES))]
_idx8_fm  = [FEATURE_COLS.index(f) for f in _top8_fm]
_means    = np.zeros((n_cls_plot, len(_top8_fm)))
for ci in range(n_cls_plot):
    mask = (y == ci)
    if mask.any():
        _means[ci] = X_raw[:, _idx8_fm][mask].mean(axis=0)
_mn           = _means.min(0); _mx = _means.max(0)
_means_norm   = (_means - _mn) / (_mx - _mn + 1e-9)
fig, (ax_hl, ax_hr) = plt.subplots(1, 2, figsize=(16, 6)); fig.patch.set_facecolor(DARK_BG)
fig.suptitle("Feature Mean Values per Class", color=TEXT_COL, fontsize=13)
style_ax(ax_hl)
_cmap2 = LinearSegmentedColormap.from_list("ids2", [DARK_BG, ACCENT2, ACCENT1], N=256)
_im2   = ax_hl.imshow(_means_norm, cmap=_cmap2, aspect="auto", vmin=0, vmax=1)
ax_hl.set_xticks(range(len(_top8_fm))); ax_hl.set_yticks(range(n_cls_plot))
ax_hl.set_xticklabels(_top8_fm, rotation=40, ha="right", color=TEXT_COL, fontsize=8)
ax_hl.set_yticklabels(cls_names, color=TEXT_COL, fontsize=9)
ax_hl.set_title("Normalised Feature Means (heatmap)", color=TEXT_COL)
for ci in range(n_cls_plot):
    for fi in range(len(_top8_fm)):
        ax_hl.text(fi, ci, f"{_means_norm[ci,fi]:.2f}",
                   ha="center", va="center", color=TEXT_COL, fontsize=7)
cb2 = plt.colorbar(_im2, ax=ax_hl, fraction=0.046, pad=0.04)
cb2.ax.yaxis.set_tick_params(color=TEXT_COL); cb2.outline.set_edgecolor(BORDER)
style_ax(ax_hr)
_bw_fm = 0.8 / n_cls_plot; _xf = np.arange(len(_top8_fm))
for ci, (cls, col) in enumerate(zip(cls_names, cls_palette)):
    off = (ci - (n_cls_plot-1)/2) * _bw_fm
    ax_hr.bar(_xf + off, _means_norm[ci], _bw_fm, label=cls, color=col, alpha=0.85)
ax_hr.set_xticks(_xf)
ax_hr.set_xticklabels(_top8_fm, rotation=40, ha="right", color=TEXT_COL, fontsize=8)
ax_hr.set_ylabel("Normalised Mean")
ax_hr.set_title("Feature Means by Class (grouped bars)", color=TEXT_COL)
ax_hr.legend(**LEG_KW)
plt.tight_layout(); savefig(fig, "feature_mean_per_class.png")

# ── PLOTS 10 & 11 — Interactive 3D HTML ──────────────────────────────────────
print("PLOT 10 — interactive_3d_classification.html ...")
print("PLOT 11 — interactive_3d_correctness.html ...")
_pca      = PCA(n_components=3, random_state=RANDOM_STATE)
_X3d      = _pca.fit_transform(X_te_k)
_var      = _pca.explained_variance_ratio_ * 100
_cls_label_arr = np.array([le.classes_[yi] for yi in y_test])
_pred_xgb_arr  = final_models.get("XGBoost", final_models.get("RF"))["pred"]
_pred_rf_arr   = final_models["RF"]["pred"]
_correct_xgb   = (_pred_xgb_arr == y_test)
_correct_rf    = (_pred_rf_arr  == y_test)

_HTML_HEAD = """<!DOCTYPE html><html lang="en"><head><meta charset="UTF-8">
<meta name="viewport" content="width=device-width,initial-scale=1.0"><title>{title}</title>
<script src="https://cdn.plot.ly/plotly-2.27.0.min.js"></script>
<style>body{{margin:0;background:#0d1117;color:#e6edf3;font-family:'Segoe UI',system-ui,sans-serif}}
h1{{text-align:center;padding:18px 0 4px;font-size:1.25rem;color:#4cc9f0}}
p.sub{{text-align:center;color:#8b949e;font-size:.82rem;margin:0 0 10px}}
#chart{{width:100vw;height:88vh}}</style></head><body>
<h1>{title}</h1><p class="sub">{subtitle}</p><div id="chart"></div><script>
"""
_HTML_FOOT = "\n</script>\n</body>\n</html>"

def _write_html(path, title, subtitle, js):
    with open(path, "w", encoding="utf-8") as fh:
        fh.write(_HTML_HEAD.format(title=title, subtitle=subtitle))
        fh.write(js)
        fh.write(_HTML_FOOT)
    print(f"  Saved → {path}")

_layout_base = {
    "paper_bgcolor": "#0d1117", "plot_bgcolor": "#161b22",
    "font": {"color": "#e6edf3", "family": "Segoe UI,system-ui"},
    "legend": {"bgcolor": "#1c2333", "bordercolor": "#30363d", "borderwidth": 1},
    "margin": {"l": 0, "r": 0, "t": 30, "b": 0},
    "scene": {
        "xaxis": {"title": f"PC1 ({_var[0]:.1f}%)", "backgroundcolor": "#161b22",
                  "gridcolor": "#21262d", "color": "#e6edf3"},
        "yaxis": {"title": f"PC2 ({_var[1]:.1f}%)", "backgroundcolor": "#161b22",
                  "gridcolor": "#21262d", "color": "#e6edf3"},
        "zaxis": {"title": f"PC3 ({_var[2]:.1f}%)", "backgroundcolor": "#161b22",
                  "gridcolor": "#21262d", "color": "#e6edf3"},
    }
}

_traces_cls = []
for cls in le.classes_:
    mask = (_cls_label_arr == cls)
    if not mask.any(): continue
    col = CLS_COLORS.get(cls, ACCENT2)
    _traces_cls.append({
        "type": "scatter3d", "name": cls,
        "x": _X3d[mask, 0].tolist(), "y": _X3d[mask, 1].tolist(), "z": _X3d[mask, 2].tolist(),
        "mode": "markers",
        "marker": {"size": 4, "color": col, "opacity": 0.82, "line": {"width": 0}},
        "hovertemplate": (f"<b>{cls}</b><br>PC1:%{{x:.2f}}<br>"
                          f"PC2:%{{y:.2f}}<br>PC3:%{{z:.2f}}<extra></extra>"),
    })
_write_html(
    "interactive_3d_classification.html",
    "3D PCA — Classification Space",
    f"Coloured by true class · PC1+PC2+PC3 explain {sum(_var):.1f}% variance",
    f"var traces={json.dumps(_traces_cls)};\nvar layout={json.dumps(_layout_base)};\n"
    "Plotly.newPlot('chart',traces,layout,{responsive:true,displayModeBar:true});\n"
)

_groups = {
    "XGB correct":   (_correct_xgb,  "#4cc9f0"),
    "XGB incorrect": (~_correct_xgb, "#e94560"),
    "RF correct":    (_correct_rf,   "#2ec4b6"),
    "RF incorrect":  (~_correct_rf,  "#f4a261"),
}
_traces_corr = []; _gkeys = list(_groups.keys())
for gn, (gm, gc) in _groups.items():
    if not gm.any(): continue
    hover = [f"True: {le.classes_[yi]}" for yi in y_test[gm]]
    _traces_corr.append({
        "type": "scatter3d", "name": gn,
        "x": _X3d[gm, 0].tolist(), "y": _X3d[gm, 1].tolist(), "z": _X3d[gm, 2].tolist(),
        "mode": "markers",
        "marker": {"size": 4, "color": gc, "opacity": 0.80,
                   "symbol": "circle" if "correct" in gn else "x"},
        "text": hover,
        "hovertemplate": "%{text}<br>PC1:%{x:.2f} PC2:%{y:.2f} PC3:%{z:.2f}<extra></extra>",
    })
_nt = len(_traces_corr)
_layout_corr = dict(_layout_base)
_layout_corr["updatemenus"] = [{
    "type": "buttons", "x": 0.02, "y": 0.98,
    "xanchor": "left", "yanchor": "top",
    "bgcolor": "#1c2333", "bordercolor": "#30363d",
    "font": {"color": "#e6edf3"},
    "buttons": [
        {"label": "Show All",  "method": "restyle", "args": [{"visible": [True]*_nt}]},
        {"label": "XGB Only",  "method": "restyle", "args": [{"visible": [i < 2 for i in range(_nt)]}]},
        {"label": "RF Only",   "method": "restyle", "args": [{"visible": [i >= 2 for i in range(_nt)]}]},
        {"label": "Errors Only","method": "restyle",
         "args": [{"visible": ["incorrect" in _gkeys[i] for i in range(_nt)]}]},
    ],
}]
_write_html(
    "interactive_3d_correctness.html",
    "3D PCA — Prediction Correctness",
    "XGBoost & RF — correct (●) vs incorrect (✕)",
    f"var traces={json.dumps(_traces_corr)};\nvar layout={json.dumps(_layout_corr)};\n"
    "Plotly.newPlot('chart',traces,layout,{responsive:true,displayModeBar:true});\n"
)


# =============================================================================
# FINAL SUMMARY
# =============================================================================
print("\n" + "=" * 70)
print("FINAL SUMMARY — v17 STRATIFIED")
print("=" * 70)
print(f"""
  PCAP           : {PCAP_PATH}
  Packets        : {total_pkts:,}
  Attacker MAC   : {ATTACKER_MAC}
  Router   MAC   : {ROUTER_MAC}

  ── SPLITTING APPROACH ──────────────────────────────────────────────────
  Method     : Stratified train_test_split (NO temporal/chronological)
  Split A    : random_state={RANDOM_STATE}   (80/20)
  Split B    : random_state={RANDOM_STATE_B}  (80/20, different partition)
  CV         : 5-Fold StratifiedKFold (full dataset)
  Imbalance  : SMOTE applied when minority < 15% of majority (train only)

  ── WINDOW CLASS DISTRIBUTION ───────────────────────────────────────────""")
for cls in ["Normal","RA_Attack","ND_Attack","Combined_Attack"]:
    n   = vc_win.get(cls, 0)
    pct = 100*n/len(df_win) if len(df_win) else 0
    print(f"    {cls:16s}: {n:5,d} windows  ({pct:.1f}%)")
if _HAS_SMOTE and needs_smote:
    print(f"\n  [SMOTE] Training expanded: {len(y_train_raw):,} → {len(y_train):,} windows")
print(f"""
  ── BEST HYPERPARAMETERS ────────────────────────────────────────────────""")
if _HAS_XGB:
    print(f"    XGBoost : LR={best_xgb_lr}, max_depth={best_xgb_depth}, gamma=1.0")
d_str = str(best_rf_depth) if best_rf_depth is not None else "None"
print(f"    RF      : n_estimators={best_rf_nest}, max_depth={d_str}")
print(f"""
  ── FEATURE-COUNT SENSITIVITY ───────────────────────────────────────────
    XGBoost best k = {best_k_xgb}
    RF      best k = {best_k_rf}
    Top features   : {TOP_K_FEATURES}

  ── FINAL RESULTS (Split A — Stratified Holdout) ────────────────────────""")
for mn, res in final_models.items():
    print(f"    {mn:10s}  Acc={res['acc_te']*100:.2f}%  BalAcc={res['bacc']*100:.2f}%  "
          f"MacF1={res['mf1']*100:.2f}%  ROC={res['roc']:.4f}  Gap={res['gap']*100:+.2f}%")
print(f"""
  ── PARTITION VARIANCE (Split A vs Split B) ─────────────────────────────""")
for mn, res in temp_models.items():
    delta = final_models.get(mn, {}).get("bacc", res["bacc"]) - res["bacc"]
    print(f"    {mn:10s}  Split-B BalAcc={res['bacc']*100:.2f}%  "
          f"MacF1={res['mf1']*100:.2f}%  Variance={delta*100:+.2f}%  "
          f"CV={res['cv_mean_bacc']*100:.2f}%±{res['cv_std_bacc']*100:.2f}%")
if comp_results:
    print(f"""
  ── MULTI-MODEL COMPARISON ──────────────────────────────────────────────
  {'Model':10s}  {'A-BalAcc':>9}  {'A-MacF1':>8}  {'B-BalAcc':>9}  {'B-MacF1':>8}  {'Variance':>9}
  {'-'*68}""")
    for mn, res in comp_results.items():
        deg = res["A"]["bacc"] - res["B"]["bacc"]
        v   = ("stable"   if abs(deg) < 0.05 else
               "moderate" if abs(deg) < 0.10 else "high-variance")
        print(f"  {mn:10s}  {res['A']['bacc']*100:>8.2f}%  {res['A']['mf1']*100:>7.2f}%  "
              f"{res['B']['bacc']*100:>8.2f}%  {res['B']['mf1']*100:>7.2f}%  "
              f"{deg*100:>+8.2f}%  {v}")
print(f"""
  ── OUTPUT FILES ────────────────────────────────────────────────────────
    {CSV_PKT_PATH}
    {CSV_WIN_PATH}
    accuracy_vs_features.png       selectkbest_ranking.png
    feature_analysis.png           iat_per_class.png
    learning_curve.png             confusion_matrices.png
    comparative_line_graphs.png    comparative_bar_graphs.png
    feature_mean_per_class.png
    interactive_3d_classification.html
    interactive_3d_correctness.html
""")
print("=" * 70)

STEP 0 — Loading node identity (nodes.csv) ...
  Attacker MAC : aa:c1:ab:68:a5:ec
  Router   MAC : aa:c1:ab:36:e8:c9
  Victim   MACs: 20
    aa:c1:ab:0a:78:ec
    aa:c1:ab:0b:ee:58
    aa:c1:ab:21:40:e8
    aa:c1:ab:25:a1:fd
    aa:c1:ab:38:d2:07
    aa:c1:ab:54:e2:5b
    aa:c1:ab:6d:43:6b
    aa:c1:ab:7a:d3:32
    aa:c1:ab:96:57:e6
    aa:c1:ab:9b:d0:54
    aa:c1:ab:ac:9c:cb
    aa:c1:ab:b9:a5:1e
    aa:c1:ab:cc:f1:a4
    aa:c1:ab:d2:79:4f
    aa:c1:ab:d5:d7:11
    aa:c1:ab:dd:2e:0a
    aa:c1:ab:e4:3a:57
    aa:c1:ab:ee:8e:cc
    aa:c1:ab:f7:8d:a1
    aa:c1:ab:fb:ac:d9

  NOTE: Attacker MAC spoofed — events.csv is sole label source.

STEP 1 — Loading events.csv and building phase timeline ...
  Phase name → label mapping:
    baseline_1                     → Normal
    baseline_final                 → Normal
    combined_attack                → Combined_Attack
    nd_attack_1                    → ND_Attack
    nd_attack_2                    → ND_Attack
    nd_attack_3                 